# Cohort retention của Olist bằng SQL trên DuckDB

Câu hỏi kinh điển của vòng SQL phỏng vấn DS: *"Viết query tính retention theo cohort tháng"*. Bài này
làm trọn câu đó trên Olist thật, kèm những chi tiết phân biệt ứng viên senior: khách là
`customer_unique_id`, phân biệt **0%** với **chưa quan sát được**, và trung bình có trọng số.

Định nghĩa dùng chung của track:
- **Đơn giao xong**: `order_status = 'delivered'` và có `order_delivered_customer_date`.
- **Khách**: `customer_unique_id` (Olist cấp `customer_id` mới cho mỗi đơn).
- Chỉ lấy đơn mua trước `2018-09-01` (sau đó là đơn còn dở dang).
- **Cohort** của khách: tháng của đơn giao xong đầu tiên. `month_index` = số tháng lịch từ cohort tới
  tháng có đơn (`DATE_DIFF('month', cohort, month)`).

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hằng và hàm.
- Cell gắn tag `explore` (dòng đầu `# explore`) đọc dữ liệu thật, vẽ, gọi hàm. Test bỏ qua các cell này.
- Chấm: `uv run pytest tracks/data-science/modules/08-coding-for-ds/exercises/intermediate/02-cohort-retention-sql -q`

In [ ]:
import duckdb
import numpy as np
import pandas as pd

In [ ]:
# explore: nạp Olist vào DuckDB dưới dạng VIEW (tải bằng: uv run python data/download_olist.py)
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data" / "raw"
con = duckdb.connect()
for table in ["orders", "customers"]:
    con.execute(f"CREATE VIEW {table} AS SELECT * FROM read_csv_auto('{RAW / f'olist_{table}_dataset.csv'}')")
con.sql("SELECT order_status, COUNT(*) AS n FROM orders GROUP BY 1 ORDER BY n DESC").df()

In [ ]:
# explore: vì sao phải dùng customer_unique_id? So số customer_id với số customer_unique_id
con.sql("SELECT COUNT(DISTINCT customer_id) AS customer_ids, COUNT(DISTINCT customer_unique_id) AS people FROM customers").df()

## 1. Đếm khách theo cohort và tháng

`cohort_counts(con)` trả DataFrame cột `cohort` (ngày đầu tháng), `month_index` (int), `customers`
(số khách **khác nhau** có đơn giao xong trong tháng đó). Một khách mua 2 lần trong cùng tháng chỉ
đếm 1. Viết bằng **một câu SQL** (CTE), không lặp trong Python.

In [ ]:
COHORT_SQL = """
WITH delivered AS (
    SELECT c.customer_unique_id, o.order_purchase_timestamp
    FROM orders o
    JOIN customers c USING (customer_id)
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_purchase_timestamp < TIMESTAMP '2018-09-01'
),
active AS (
    SELECT DISTINCT customer_unique_id,
           DATE_TRUNC('month', order_purchase_timestamp)::DATE AS month
    FROM delivered
),
cohort AS (
    SELECT customer_unique_id, MIN(month) AS cohort
    FROM active
    GROUP BY 1
)
SELECT c.cohort,
       DATE_DIFF('month', c.cohort, a.month) AS month_index,
       COUNT(*) AS customers               -- active đã DISTINCT theo (khách, tháng)
FROM active a
JOIN cohort c USING (customer_unique_id)
GROUP BY 1, 2
ORDER BY 1, 2
"""


def cohort_counts(con: duckdb.DuckDBPyConnection) -> pd.DataFrame:
    """Cột cohort (datetime), month_index (int), customers (int), sắp theo cohort, month_index."""
    out = con.sql(COHORT_SQL).df()
    return out.assign(
        cohort=pd.to_datetime(out["cohort"]),
        month_index=out["month_index"].astype("int64"),
        customers=out["customers"].astype("int64"),
    )

In [ ]:
# explore
counts = cohort_counts(con)
counts.head(10)

## 2. Ma trận retention: 0% khác "chưa biết"

`retention_matrix(counts, last_month)`: index là cohort, cột là `month_index` từ 0 tới số tháng lớn
nhất quan sát được, giá trị là `customers / cỡ cohort`.

- Ô cohort `c`, tháng `k` mà `c + k` tháng **sau** `last_month` (tháng cuối có dữ liệu): `NaN`, vì
  chưa quan sát được.
- Ô đã quan sát được mà không có ai quay lại: `0.0`, không phải `NaN`.

Lỗi hay gặp: pivot xong để nguyên NaN cho cả hai trường hợp, rồi `mean()` theo cột bỏ qua NaN, làm
retention bị thổi phồng ở các cohort nhỏ.

In [ ]:
def _months_observed(cohorts, last_month: str) -> np.ndarray:
    """Số tháng sau cohort mà dữ liệu đã bao phủ (cohort = last_month -> 0)."""
    last = pd.Period(pd.Timestamp(last_month), "M")
    return np.array([(last - pd.Period(c, "M")).n for c in cohorts])


def retention_matrix(counts: pd.DataFrame, last_month: str) -> pd.DataFrame:
    """Index cohort, cột month_index 0..K, giá trị tỉ lệ; NaN khi chưa quan sát được."""
    size = counts.loc[counts["month_index"] == 0].set_index("cohort")["customers"]
    observed = _months_observed(size.index, last_month)
    mat = (counts.pivot(index="cohort", columns="month_index", values="customers")
                 .reindex(index=size.index, columns=range(observed.max() + 1))
                 .fillna(0))
    rate = mat.div(size, axis=0)
    k = np.arange(rate.shape[1])
    rate = rate.mask(k[None, :] > observed[:, None])      # chưa quan sát được -> NaN
    rate.columns.name = "month_index"
    return rate


def average_retention(counts: pd.DataFrame, k: int, last_month: str) -> float:
    """Retention tháng k trung bình CÓ TRỌNG SỐ theo cỡ cohort, chỉ trên cohort đã quan sát được tháng k."""
    size = counts.loc[counts["month_index"] == 0].set_index("cohort")["customers"]
    eligible = size[_months_observed(size.index, last_month) >= k]
    returned = counts.loc[(counts["month_index"] == k) & counts["cohort"].isin(eligible.index), "customers"].sum()
    return float(returned / eligible.sum())

In [ ]:
# explore: heatmap retention tháng 1..12 (bỏ cột 0 luôn là 100%)
import matplotlib.pyplot as plt
import seaborn as sns

mat = retention_matrix(counts, "2018-08-01")
mat.index = mat.index.strftime("%Y-%m")
fig, ax = plt.subplots(figsize=(11, 7))
sns.heatmap(mat.loc["2017-01":, 1:12] * 100, annot=True, fmt=".2f", cmap="Blues", cbar_kws={"label": "%"}, ax=ax)
ax.set_title("Olist: % khách của cohort có đơn giao xong ở tháng thứ k")
plt.show()

In [ ]:
# explore: so trung bình có trọng số với trung bình cộng thô của cột (cohort 2016-12 chỉ có 1 khách!)
for k in [1, 2, 3, 6, 12]:
    print(k, round(average_retention(counts, k, "2018-08-01") * 100, 3), round(mat[k].mean() * 100, 3))

## 3. Tỉ lệ khách mua lại

`repeat_customer_rate(con)`: trong số khách có ít nhất một đơn giao xong (mua trước 2018-09-01), tỉ
lệ khách có **từ 2 đơn giao xong trở lên** (đếm theo đơn, kể cả 2 đơn trong cùng tháng). Một câu SQL.

In [ ]:
def repeat_customer_rate(con: duckdb.DuckDBPyConnection) -> float:
    sql = """
    SELECT AVG((n >= 2)::INT)
    FROM (
        SELECT c.customer_unique_id, COUNT(*) AS n
        FROM orders o
        JOIN customers c USING (customer_id)
        WHERE o.order_status = 'delivered'
          AND o.order_delivered_customer_date IS NOT NULL
          AND o.order_purchase_timestamp < TIMESTAMP '2018-09-01'
        GROUP BY 1
    )
    """
    return float(con.execute(sql).fetchone()[0])

In [ ]:
# explore
repeat_customer_rate(con)

## 4. Kết luận cho stakeholder (bài mẫu)

**Olist gần như là một doanh nghiệp "mua một lần".** Chỉ 3.0% khách có từ 2 đơn giao xong trở lên;
retention tháng 1 trung bình (có trọng số) là 0.48%, và từ tháng 3 trở đi chỉ còn khoảng 0.2–0.3%
mỗi tháng. Heatmap không cho thấy cohort gần đây giữ khách tốt hơn cohort cũ.

Hệ quả: tăng trưởng hiện đến gần như hoàn toàn từ khách mới, nên chi phí thu hút khách (CAC) phải
được bù bằng **một** đơn hàng. Trước khi kết luận "Olist giữ khách kém", cần lưu ý: Olist bán qua
các sàn đối tác, khách quay lại có thể mua qua kênh khác mà dữ liệu này không thấy; bộ dữ liệu công
khai cũng chỉ là một mẫu. Bước tiếp theo: so retention theo danh mục (hàng tiêu dùng lặp lại vs đồ
bền) và theo trải nghiệm đơn đầu (giao trễ, review xấu) để tìm đòn bẩy giữ khách.